In [1]:
import dotenv
%load_ext dotenv
%dotenv

import os
import platform
import json
import random
import time

import jax
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from absl import app, flags
from ml_collections import config_flags

from agents import agents
from envs.env_utils import make_env_and_datasets
from utils.datasets import Dataset, ReplayBuffer
from utils.evaluation import evaluate, flatten, supply_rng
from utils.flax_utils import restore_agent, save_agent

import jax
import numpy as np
import matplotlib.pyplot as plt

def supply_rng(f, rng=jax.random.PRNGKey(0)):
    """Helper function to split the random number generator key before each call to the function."""

    def wrapped(*args, **kwargs):
        nonlocal rng
        rng, key = jax.random.split(rng)
        return f(*args, seed=key, **kwargs)

    return wrapped

# Configuration
env_name = "antmaze-large-navigate-singletask-task2-v0"
seed = 0
buffer_size = 2000000
eval_episodes = 50

# Agent paths and epochs
a5_jacb_reg_e5_agent_path="/mnt/nas/jaehyeok/fql/exp/fql/Debug/iql_antmaze-large-navigate-singletask-task2-v0_sd000_20250904_173407_utd-ratio1"
# a3_jacb_reg_e5_agent_path="/mnt/nas/jaehyeok/fql/exp/fql/Debug/iql_antmaze-large-navigate-singletask-task2-v0_sd000_20250904_174856_utd-ratio1"

agent_path=a5_jacb_reg_e5_agent_path

restore_epoch=1000000

# Make environment and datasets
env, eval_env, train_dataset, val_dataset = make_env_and_datasets(env_name, frame_stack=None)
train_dataset = Dataset.create(**train_dataset)
train_dataset = ReplayBuffer.create_from_initial_dataset(
    dict(train_dataset), size=max(buffer_size, train_dataset.size + 1)
)

# Initialize random seeds
random.seed(seed)
np.random.seed(seed)
example_batch = train_dataset.sample(1)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/home/jaehyeok/miniconda3/envs/fql/lib/python3.10/site-packages/Cython/Distutils/old_build_ext.py:14: DeprecationWarning: dep_util is Deprecated. Use functions from setuptools instead.
  from distutils.dep_util import newer, newer_group
/home/jaehyeok/miniconda3/envs/fql/lib/python3.10/site-packages/Cython/Distutils/old_build_ext.py:14: DeprecationWarning: dep_util is Deprecated. Use functions from setuptools instead.
  from distutils.dep_util import newer, newer_group
<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead


In [2]:
def load_agent(agent_path, restore_epoch, agent_config, example_batch, seed):
    agent_class = agents[agent_config['agent_name']]
    agent = agent_class.create(
        seed,
        example_batch['observations'],
        example_batch['actions'],
        agent_config,
    )
    agent = restore_agent(agent, agent_path, restore_epoch)
    return agent

In [3]:
restore_epoch=1000000

a5_jacb_reg_e5_agent_path="/mnt/nas/jaehyeok/fql/exp/fql/Debug/iql_antmaze-large-navigate-singletask-task2-v0_sd000_20250904_173407_utd-ratio1"
a3_jacb_reg_e5_agent_path="/mnt/nas/jaehyeok/fql/exp/fql/Debug/iql_antmaze-large-navigate-singletask-task2-v0_sd000_20250904_174856_utd-ratio1"
a3_jacb_reg_e6_agent_path="/mnt/nas/jaehyeok/fql/exp/fql/Debug/iql_antmaze-large-navigate-singletask-task2-v0_sd000_20250904_200256_utd-ratio1"
a3_jacb_agent_path="/mnt/nas/jaehyeok/fql/exp/fql/Debug/iql_antmaze-large-navigate-singletask-task2-v0_sd000_20250904_195824_utd-ratio1"

In [4]:
from agents.iql import get_config as get_iql_config

a5_jacb_reg_e5_agent=load_agent(a5_jacb_reg_e5_agent_path, 1000000, get_iql_config(), example_batch, seed)
a3_jacb_reg_e5_agent=load_agent(a3_jacb_reg_e5_agent_path, 1000000, get_iql_config(), example_batch, seed)
a3_jacb_reg_e6_agent=load_agent(a3_jacb_reg_e6_agent_path, 1000000, get_iql_config(), example_batch, seed)
a3_jacb_agent=load_agent(a3_jacb_agent_path, 1000000, get_iql_config(), example_batch, seed)

Restored from /mnt/nas/jaehyeok/fql/exp/fql/Debug/iql_antmaze-large-navigate-singletask-task2-v0_sd000_20250904_173407_utd-ratio1/params_1000000.pkl
Restored from /mnt/nas/jaehyeok/fql/exp/fql/Debug/iql_antmaze-large-navigate-singletask-task2-v0_sd000_20250904_174856_utd-ratio1/params_1000000.pkl
Restored from /mnt/nas/jaehyeok/fql/exp/fql/Debug/iql_antmaze-large-navigate-singletask-task2-v0_sd000_20250904_200256_utd-ratio1/params_1000000.pkl
Restored from /mnt/nas/jaehyeok/fql/exp/fql/Debug/iql_antmaze-large-navigate-singletask-task2-v0_sd000_20250904_195824_utd-ratio1/params_1000000.pkl


In [22]:
def evaluate_from_init_state(actor_fn, eval_env, init_state, perturbed_agent_body, num_trials, eval_timestep, qpos_dim=15, eval_temperature=0.0):
    qvel_dim = init_state.shape[0] - qpos_dim
    avg_rewards = []
    terminated_timesteps = []
    success = 0

    init_state[2:] = perturbed_agent_body
    for _ in tqdm(range(num_trials)):
        eval_env.reset()
        init_ob = init_state
        eval_env.set_state(init_ob[:qpos_dim], init_ob[qpos_dim:])

        ob = init_ob

        rewards = []
        
        for i in range(eval_timestep):
            action = actor_fn(observations=ob, temperature=eval_temperature)
            ob, reward, terminated, truncated, info = eval_env.step(action)
            rewards.append(reward)
            if terminated:
                terminated_timesteps.append(i)
                if info['success']:
                    success += 1
                break
        avg_rewards.append(np.array(rewards).mean())

    return success / num_trials, np.array(avg_rewards).mean(), np.array(terminated_timesteps).mean()

In [23]:
def get_init_state(eval_env, x=None, y=None):
    if x is None and y is None:
        init_state, _ = eval_env.reset()
    else:
        init_state, _ = eval_env.reset()
        init_state[0] = x
        init_state[1] = y
    return init_state

# init_x = 36
# init_y = 23.5

# get_init_state(eval_env, init_x, init_y)

# eval_agent = a5_jacb_reg_e5_agent

# evaluate_from_init_state(
#     supply_rng(eval_agent.sample_actions, rng=jax.random.PRNGKey(np.random.randint(0, 2**32))),
#     eval_env,
#     init_state,
#     num_trials=10,
#     eval_timestep=1000,
#     qpos_dim=15,
#     eval_temperature=0.0
# )

In [28]:
agent_lists = {
    "a5_jacb_reg_e5_agent": a5_jacb_reg_e5_agent,
    "a3_jacb_reg_e5_agent": a3_jacb_reg_e5_agent,
    "a3_jacb_reg_e6_agent": a3_jacb_reg_e6_agent,
    "a3_jacb_agent": a3_jacb_agent
}

rng = jax.random.PRNGKey(np.random.randint(0, 2**32))

init_x_list = [36, 36, 27.5, 12]
init_y_list = [23.5, 0., 7.5, 0.]
init_state, _ = eval_env.reset()
perturbation_range = 0.05
perturbed_agend_bodies = []

agent_body_ob = init_state[2:]

exp_rng, *state_rngs = jax.random.split(rng, len(init_x_list) + 1)

for i in range(len(init_x_list)):
    perturbed_agnent_body = agent_body_ob + jax.random.uniform(state_rngs[i], agent_body_ob.shape, minval=-perturbation_range, maxval=perturbation_range)
    perturbed_agend_bodies.append(perturbed_agnent_body)


# use the variable name as the key
rets = {x:{} for x in agent_lists.keys()}

for agent_name, agent in tqdm(agent_lists.items()):
    idx = 0
    for init_x, init_y in zip(init_x_list, init_y_list):
        init_state = get_init_state(eval_env, init_x, init_y)
        success, avg_reward, avg_terminated_timesteps = evaluate_from_init_state(
            supply_rng(agent.sample_actions, rng=exp_rng),
            eval_env,
            init_state,
            perturbed_agend_bodies[idx],
            num_trials=50,
            eval_timestep=1000,
            qpos_dim=15,
            eval_temperature=0.0
        )
        rets[agent_name][(init_x, init_y)] = (success, avg_reward, avg_terminated_timesteps)
        idx += 1


  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:49<00:00,  1.01it/s]
/tmp/ipykernel_2363093/4129331006.py:28: RuntimeWarning: Mean of empty slice.
  return success / num_trials, np.array(avg_rewards).mean(), np.array(terminated_timesteps).mean()
100%|██████████| 4/4 [08:50<00:00, 132.66s/it]


In [26]:
for agent_name, agent_rets in rets.items():
    success = []
    rewards = []
    terminated_timesteps = []
    print(agent_name)
    for init_x, init_y in agent_rets.keys():
        exp_success = np.array(agent_rets[(init_x, init_y)][0]).mean()
        exp_reward = np.array(agent_rets[(init_x, init_y)][1]).mean()
        exp_terminated_timesteps = np.array(agent_rets[(init_x, init_y)][2]).mean()
        success.append(exp_success)
        rewards.append(exp_reward)
        terminated_timesteps.append(exp_terminated_timesteps)
        print(f"({init_x}, {init_y}): {exp_success}, {exp_reward}, {exp_terminated_timesteps}")
    success = np.array(success).mean()
    rewards = np.array(rewards).mean()
    terminated_timesteps = np.array(terminated_timesteps).mean()
    print(f"Total: {success}, {rewards}, {terminated_timesteps}")

a5_jacb_reg_e5_agent
(36, 23.5): 0.4, -0.9994781432397477, 794.25
(36, 0.0): 0.9, -0.9982235295392131, 511.8888888888889
(27.5, 7.5): 0.3, -0.9996272876628787, 821.0
(12, 0.0): 1.0, -0.997201421822763, 359.5
Total: 0.65, -0.9986325955661506, 621.6597222222222
a3_jacb_reg_e5_agent
(36, 23.5): 0.9, -0.9983486907138623, 554.8888888888889
(36, 0.0): 0.8, -0.9982076591639253, 446.0
(27.5, 7.5): 1.0, -0.9984692625644342, 670.0
(12, 0.0): 1.0, -0.997302712575825, 381.1
Total: 0.925, -0.9980820812545117, 512.9972222222223
a3_jacb_reg_e6_agent
(36, 23.5): 0.8, -0.9985185474049448, 546.75
(36, 0.0): 0.8, -0.998268638601508, 463.5
(27.5, 7.5): 0.9, -0.9984973598913103, 599.1111111111111
(12, 0.0): 0.6, -0.9981144510458894, 319.1666666666667
Total: 0.775, -0.9983497492359131, 482.13194444444446
a3_jacb_agent
(36, 23.5): 0.6, -0.9992072759288757, 758.5
(36, 0.0): 0.8, -0.9983880115436549, 496.625
(27.5, 7.5): 1.0, -0.9979599202182202, 494.6
(12, 0.0): 1.0, -0.9970484574519934, 339.1
Total: 0.85, -0